# Первая практика — мини-анализ HMDA

## Подготовка

In [1]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: C:\Users\Артёмка\анализ данных\data-analysis-course\.venv\Scripts\python.exe
pandas: 2.2.2
Рабочая папка: C:\Users\Артёмка\анализ данных\data-analysis-course\lessons\01-introduction\work


## Маленький пример reading_example.csv

In [2]:
example = pd.read_csv(
    DATA / "reading_example.csv",
    dtype={"lei": "string", "county_code": "string"},
)
example

,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,275000.0,72.0
1,3,2023,549300P1347BFCBHJV60,11001,1,585000.0,174.0
2,278,2023,KB1H1DSPRFMYMCUFXT09,11001,2,475000.0,108.0
3,304,2023,549300VZVN841I2ILS84,11001,2,905000.0,353.0
4,21,2023,549300RPOGWJRH63HS39,11001,3,595000.0,184.0
5,27,2023,JJKC32MCHWDI71265Z06,11001,3,1805000.0,667.0


## A. Разделитель CSV

In [3]:
wrong_sep = pd.read_csv(
    DATA / "reading_example.csv",
    sep=";",
)
print(wrong_sep.shape)
print(example.shape)
wrong_sep

(6, 1)
(6, 7)


,"source_row,activity_year,lei,county_code,action_taken,loan_amount,income"
0,"2,2023,549300VZVN841I2ILS84,11001,1,275000.0,72.0"
1,"3,2023,549300P1347BFCBHJV60,11001,1,585000.0,1..."
2,"278,2023,KB1H1DSPRFMYMCUFXT09,11001,2,475000.0..."
3,"304,2023,549300VZVN841I2ILS84,11001,2,905000.0..."
4,"21,2023,549300RPOGWJRH63HS39,11001,3,595000.0,..."
5,"27,2023,JJKC32MCHWDI71265Z06,11001,3,1805000.0..."


## B. Частичное чтение

In [4]:
preview = pd.read_csv(DATA / "reading_example.csv", nrows=3)
small_columns = pd.read_csv(
    DATA / "reading_example.csv",
    usecols=["action_taken", "income"],
)

print("preview shape:", preview.shape)
print("small_columns shape:", small_columns.shape)

preview shape: (3, 7)
small_columns shape: (6, 2)


## Выбор Series и DataFrame

In [5]:
outcomes = example["action_taken"]
print(type(outcomes))
outcomes

<class 'pandas.core.series.Series'>


0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: int64

## Выбор нескольких столбцов

In [6]:
names = ["source_row", "action_taken", "income"]
selected = example[names]
print(type(selected))
selected.head(6)

<class 'pandas.core.frame.DataFrame'>


,source_row,action_taken,income
0,2,1,72.0
1,3,1,174.0
2,278,2,108.0
3,304,2,353.0
4,21,3,184.0
5,27,3,667.0


## C. Выбор столбцов и просмотр

In [7]:
outcomes = example["action_taken"]
selected = example[["action_taken"]]
print(type(outcomes), type(selected))

amount_view = example[["source_row", "loan_amount"]].head(2)
print(amount_view)
print(example.shape)
print(example.columns)
print(example.info)

<class 'pandas.core.series.Series'> <class 'pandas.core.frame.DataFrame'>
   source_row  loan_amount
0           2     275000.0
1           3     585000.0
(6, 7)
Index(['source_row', 'activity_year', 'lei', 'county_code', 'action_taken',
       'loan_amount', 'income'],
      dtype='object')
<bound method DataFrame.info of    source_row  activity_year                   lei county_code  action_taken  \
0           2           2023  549300VZVN841I2ILS84       11001             1   
1           3           2023  549300P1347BFCBHJV60       11001             1   
2         278           2023  KB1H1DSPRFMYMCUFXT09       11001             2   
3         304           2023  549300VZVN841I2ILS84       11001             2   
4          21           2023  549300RPOGWJRH63HS39       11001             3   
5          27           2023  JJKC32MCHWDI71265Z06       11001             3   

   loan_amount  income  
0     275000.0    72.0  
1     585000.0   174.0  
2     475000.0   108.0  
3     905000.0

## Пример расчёта одобрений

In [8]:
approved_flag = outcomes.isin([1, 2])
approved_flag

0     True
1     True
2     True
3     True
4    False
5    False
Name: action_taken, dtype: bool

## Количество и доля одобрений

In [9]:
n_approved = approved_flag.sum()
n_decisions = len(example)
assert n_decisions > 0
small_approval_rate = n_approved / n_decisions
print(n_approved, n_decisions)
print(f"Одобрено: {100 * small_approval_rate:.2f}%")

4 6
Одобрено: 66.67%


## D. Доля выдач на маленьком примере

In [10]:
issued_flag = outcomes.eq(1)
n_issued = issued_flag.sum()
small_origination_rate = n_issued / n_decisions
print(issued_flag)
print(n_issued)
print(n_decisions)
print(f"Выдано: {100 * small_origination_rate}%")

0     True
1     True
2    False
3    False
4    False
5    False
Name: action_taken, dtype: bool
2
6
Выдано: 33.33333333333333%


## Основной набор decisions.csv

In [11]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
full_outcomes = decisions["action_taken"]
full_n = len(decisions)
print(decisions.shape)
full_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## E. Две доли на основном наборе

In [12]:
full_approved = full_outcomes.isin([1, 2]).sum()
full_issued = full_outcomes.eq(1).sum()
full_approval_rate = full_approved / full_n
full_origination_rate = full_issued / full_n
print(f'Числитель : {full_approved}')
print(f'Знаменатель : {full_n}')
print(f'Процент: {full_approval_rate * 100}')
print(f'Одобрено {full_approved}/{full_n}')

print(f'Числитель : {full_issued}')
print(f'Знаменатель : {full_n}')
print(f'Процент: {full_origination_rate * 100}')
print(f'Выдано {full_issued}/{full_n}')

Числитель : 6139
Знаменатель : 6830
Процент: 89.88286969253294
Одобрено 6139/6830
Числитель : 5956
Знаменатель : 6830
Процент: 87.203513909224
Выдано 5956/6830


## Проверка входных данных

In [13]:
assert full_n > 0
assert full_outcomes.isna().sum() == 0
assert full_outcomes.isin([1, 2, 3]).sum() == full_n
income_missing = decisions["income"].isna().sum()
print("Пропущен доход:", income_missing)

Пропущен доход: 342


## F. Баланс и диапазоны

In [14]:
full_denied = full_outcomes.eq(3).sum()
assert full_approved + full_denied == full_n
assert 0 <= full_issued <= full_approved <= full_n
assert 0 <= full_origination_rate <= full_approval_rate <= 1
print(f'Отказ: {full_denied}')

Отказ: 691


## G. Расчёт из отчёта

In [15]:
approval_rate = full_outcomes.eq(1).sum() / full_n
approval_rate

0.8720351390922401

## G. Исправление условия

In [16]:
origination_rate = full_outcomes.eq(1).sum() / full_n
print(f'Выдано :{origination_rate*100}')

Выдано :87.203513909224


## G. Исправление названия

In [17]:
correct_approval_rate = full_outcomes.isin([1, 2]).sum() / full_n
print(f"Одобрено: {100 * correct_approval_rate}%")

Одобрено: 89.88286969253294%
